# @property, @classmethod, and @staticmethod

## Overview
These three built-in decorators control **how methods relate to instances and classes**. They exist to solve real design problems:
- `@property`: expose computed values as attributes while hiding implementation details and enabling validation
- `@classmethod`: provide alternative constructors and class-level operations without needing an instance
- `@staticmethod`: group utility functions with a class without polluting the module namespace

Understanding these deeply lets you design APIs that are clean, safe, and intuitive.

## Table of Contents
1. `@property` — Computed Attributes
2. `@property.setter` — Validation on Assignment
3. `@property.deleter`
4. Read-Only Properties
5. `@classmethod` — Alternative Constructors
6. `@staticmethod` — Utility Functions
7. When to Use Each
8. Combining with Inheritance
9. Common Mistakes
10. Exercises
11. Mini-Project: Temperature Converter

## 1. `@property` — Computed Attributes

**The problem `@property` solves:** You start with a public attribute `self.name`. Later you need to add validation or computation. If you change it to `self.get_name()`, all callers break.

`@property` lets you change the *implementation* (from stored value to computed value, with validation) while keeping the *interface* the same — callers still use `obj.name`, not `obj.name()`.

This is the **Uniform Access Principle** in action.

In [ ]:
# Without @property: direct attribute access
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height
        self.area = width * height   # problem: stale if width/height changes!

r = Rectangle(4, 5)
print(r.area)   # 20 — correct
r.width = 10
print(r.area)   # still 20 — STALE! Bug!

print("---")

# With @property: area is always fresh
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    @property
    def area(self):   # looks like an attribute, behaves like a method
        return self.width * self.height

    @property
    def perimeter(self):
        return 2 * (self.width + self.height)

r = Rectangle(4, 5)
print(r.area)       # 20 — note: no parentheses!
r.width = 10
print(r.area)       # 50 — automatically updated!

## 2. `@property.setter` — Validation on Assignment

The setter intercepts assignments (`obj.x = value`) so you can validate, transform, or trigger side effects. Use a private `_x` attribute internally and expose `x` as the public interface.

In [ ]:
class Person:
    def __init__(self, name, age):
        self.name = name    # goes through setter
        self.age = age      # goes through setter

    @property
    def name(self):
        return self._name

    @name.setter
    def name(self, value):
        if not isinstance(value, str):
            raise TypeError(f"name must be str, got {type(value).__name__}")
        value = value.strip()
        if not value:
            raise ValueError("name cannot be empty")
        self._name = value.title()   # auto-capitalize

    @property
    def age(self):
        return self._age

    @age.setter
    def age(self, value):
        if not isinstance(value, int):
            raise TypeError(f"age must be int, got {type(value).__name__}")
        if not 0 <= value <= 150:
            raise ValueError(f"age {value} is out of range [0, 150]")
        self._age = value

    def __repr__(self):
        return f"Person(name={self._name!r}, age={self._age})"

p = Person("alice", 30)
print(p)           # Person(name='Alice', age=30)  — auto-capitalized!

p.age = 31         # works fine
print(p.age)

try:
    p.age = -5     # validation triggers
except ValueError as e:
    print(f"ValueError: {e}")

try:
    p.name = ""   # validation triggers
except ValueError as e:
    print(f"ValueError: {e}")

## 3. `@property.deleter`

The deleter intercepts `del obj.x`. Useful for cleanup, invalidating caches, or preventing deletion.

In [ ]:
class CachedComputation:
    def __init__(self, data):
        self._data = data
        self._cached_result = None

    @property
    def result(self):
        if self._cached_result is None:
            print("  [computing expensive result...]")
            self._cached_result = sum(x**2 for x in self._data)
        return self._cached_result

    @result.deleter
    def result(self):
        print("  [cache cleared]")
        self._cached_result = None

c = CachedComputation(range(100))
print(c.result)    # computes
print(c.result)    # cached
del c.result       # clear cache
print(c.result)    # recomputes

## 4. Read-Only Properties

A property without a setter is read-only. Trying to set it raises `AttributeError`. This is perfect for values that should never change after construction.

In [ ]:
import uuid

class Entity:
    def __init__(self, name):
        self.name = name
        self._id = str(uuid.uuid4())   # assigned once, in __init__

    @property
    def id(self):
        """Unique ID — read-only after creation."""
        return self._id

e = Entity("Alice")
print(e.id)    # some UUID

try:
    e.id = "new-id"   # attempt to modify read-only property
except AttributeError as ex:
    print(f"AttributeError: {ex}")

## 5. `@classmethod` — Alternative Constructors and Class-Level Operations

`@classmethod` receives the **class** as its first argument (`cls`), not an instance. The key use case: **alternative constructors** (also called factory methods).

Why `cls` instead of hardcoding the class name? Because subclasses will inherit the method and `cls` will correctly refer to the subclass.

In [ ]:
from datetime import date

class Employee:
    raise_amount = 1.04   # class-level attribute

    def __init__(self, first, last, salary):
        self.first = first
        self.last = last
        self.salary = salary

    def apply_raise(self):
        self.salary = int(self.salary * self.raise_amount)

    @classmethod
    def set_raise_amount(cls, amount):
        """Change raise amount for ALL employees (class-level)."""
        cls.raise_amount = amount

    @classmethod
    def from_string(cls, emp_str):
        """Alternative constructor: parse 'First-Last-Salary' string."""
        first, last, salary = emp_str.split("-")
        return cls(first, last, int(salary))   # cls() instead of Employee()

    @classmethod
    def from_dict(cls, data: dict):
        """Alternative constructor from a dictionary."""
        return cls(data["first"], data["last"], data["salary"])

    def __repr__(self):
        return f"Employee({self.first} {self.last}, ${self.salary:,})"

# Standard construction
emp1 = Employee("John", "Doe", 50000)

# Alternative constructors — no need to know internal structure
emp2 = Employee.from_string("Jane-Smith-60000")
emp3 = Employee.from_dict({"first": "Bob", "last": "Jones", "salary": 55000})

print(emp1)
print(emp2)
print(emp3)

# Class-level modification affects all instances
Employee.set_raise_amount(1.05)
emp1.apply_raise()
print(f"After raise: {emp1}")

In [ ]:
# Why cls matters: inheritance
class Manager(Employee):
    def __init__(self, first, last, salary, reports=None):
        super().__init__(first, last, salary)
        self.reports = reports or []

    def __repr__(self):
        return f"Manager({self.first} {self.last}, ${self.salary:,}, reports={len(self.reports)})"

# from_string is inherited. cls = Manager, so it creates a Manager!
mgr = Manager.from_string("Carol-White-80000")
print(type(mgr))   # <class '__main__.Manager'> — correct!
print(mgr)

# If from_string had used Employee() instead of cls(), it would create an Employee,
# not a Manager — that's the bug @classmethod prevents.

## 6. `@staticmethod` — Utility Functions

`@staticmethod` is a function that lives in a class's namespace but receives **no implicit first argument** (no `self`, no `cls`). It's essentially a regular function that's logically related to the class.

Use it when the function doesn't need to access class or instance state, but conceptually belongs to the class.

In [ ]:
class MathHelper:
    """A collection of math utilities — no state needed."""

    @staticmethod
    def is_prime(n):
        """Check if n is prime."""
        if n < 2:
            return False
        for i in range(2, int(n**0.5) + 1):
            if n % i == 0:
                return False
        return True

    @staticmethod
    def clamp(value, min_val, max_val):
        """Clamp value to [min_val, max_val] range."""
        return max(min_val, min(max_val, value))

    @staticmethod
    def gcd(a, b):
        """Greatest common divisor (Euclidean algorithm)."""
        while b:
            a, b = b, a % b
        return a

# Can be called on the class OR on an instance
print(MathHelper.is_prime(17))     # True
print(MathHelper.clamp(150, 0, 100))  # 100
print(MathHelper.gcd(48, 18))      # 6

mh = MathHelper()
print(mh.is_prime(23))  # also works on instance

## 7. When to Use Each

| Situation | Use |
|---|---|
| Attribute that should be computed fresh each access | `@property` |
| Validate or transform a value on assignment | `@property.setter` |
| Make an attribute read-only | `@property` (no setter) |
| Alternative constructor / parse from string/dict/file | `@classmethod` |
| Operation on class-level state, not instance state | `@classmethod` |
| Utility function logically belonging to a class | `@staticmethod` |
| Function with no access to class or instance | `@staticmethod` |

In [ ]:
# Comparison table in code
class Demo:
    class_var = "class level"

    def __init__(self):
        self.instance_var = "instance level"

    def regular_method(self):
        # Has self — can access instance AND class
        return f"{self.instance_var}, {self.class_var}"

    @classmethod
    def class_method(cls):
        # Has cls — can access class, NOT instance
        return f"{cls.class_var}"  # no self.instance_var

    @staticmethod
    def static_method():
        # No self, no cls — pure function
        return "no class or instance access"

d = Demo()
print(d.regular_method())
print(Demo.class_method())
print(Demo.static_method())

## 8. Combining with Inheritance

In [ ]:
class Animal:
    def __init__(self, name, sound):
        self._name = name
        self._sound = sound

    @property
    def name(self):
        return self._name

    @property
    def sound(self):
        return self._sound

    @classmethod
    def from_dict(cls, data):
        return cls(data["name"], data["sound"])

    @staticmethod
    def is_valid_name(name):
        return isinstance(name, str) and len(name) > 0

class Dog(Animal):
    def __init__(self, name, breed):
        super().__init__(name, "Woof")
        self._breed = breed

    @property
    def breed(self):
        return self._breed

    @classmethod
    def from_dict(cls, data):
        # Override to handle extra field
        return cls(data["name"], data.get("breed", "Unknown"))

# from_dict correctly returns Dog, not Animal
fido = Dog.from_dict({"name": "Fido", "breed": "Labrador"})
print(type(fido))           # <class '__main__.Dog'>
print(fido.name)            # Fido
print(fido.sound)           # Woof (inherited)
print(fido.breed)           # Labrador
print(Dog.is_valid_name("Rex"))  # True — static method inherited

## 9. Common Mistakes

In [ ]:
# MISTAKE 1: Infinite recursion — property using same name as itself
class Bad:
    @property
    def value(self):
        return self.value   # WRONG: calls itself recursively!
        # Correct: return self._value (use underscore for storage)

# CORRECT pattern:
class Good:
    def __init__(self, value):
        self._value = value   # private storage

    @property
    def value(self):          # public interface
        return self._value    # returns _value, not value

    @value.setter
    def value(self, v):
        self._value = v       # stores to _value

g = Good(10)
print(g.value)
g.value = 20
print(g.value)

In [ ]:
# MISTAKE 2: Using @staticmethod when @classmethod is needed (breaks inheritance)
class BaseCreator:
    @staticmethod
    def create_bad():
        return BaseCreator()   # hardcoded — subclasses get Base instances!

    @classmethod
    def create_good(cls):
        return cls()           # polymorphic — subclasses get their own type

class SubCreator(BaseCreator):
    pass

print(type(SubCreator.create_bad()))   # <class 'BaseCreator'>  WRONG
print(type(SubCreator.create_good()))  # <class 'SubCreator'>   CORRECT

In [ ]:
# MISTAKE 3: Expensive computation in @property without caching
import time

class Naive:
    def __init__(self, data):
        self._data = data

    @property
    def result(self):
        time.sleep(0.1)  # simulating expensive computation
        return sum(self._data)

# Solution: cache the result, invalidate when data changes
class Smart:
    def __init__(self, data):
        self._data = data
        self._result_cache = None

    @property
    def data(self):
        return self._data

    @data.setter
    def data(self, value):
        self._data = value
        self._result_cache = None   # invalidate cache when data changes

    @property
    def result(self):
        if self._result_cache is None:
            self._result_cache = sum(self._data)   # only compute once
        return self._result_cache

s = Smart([1, 2, 3, 4, 5])
print(s.result)   # computes
print(s.result)   # cached
s.data = [10, 20, 30]
print(s.result)   # recomputes (cache was invalidated)

## 10. Exercises

**Exercise 1 (Easy):** Create a `Circle` class with a `radius` property that validates the value is positive. Add read-only `area` and `diameter` properties.

**Exercise 2 (Medium):** Create a `BankAccount` class with a `balance` property (read-only, can't be set directly). Add `deposit(amount)` and `withdraw(amount)` methods. Add a `@classmethod from_opening_deposit(cls, amount)` constructor.

**Exercise 3 (Hard):** Create a `Vector` class with `x` and `y` properties. Add a `magnitude` read-only property, a `normalized` read-only property, and `@classmethod from_polar(cls, r, theta)` and `@staticmethod dot(v1, v2)` methods. Override `__add__`, `__mul__`, and `__repr__`.

In [ ]:
# Exercise 1 Solution
import math

class Circle:
    def __init__(self, radius):
        self.radius = radius   # goes through setter

    @property
    def radius(self):
        return self._radius

    @radius.setter
    def radius(self, value):
        if value <= 0:
            raise ValueError(f"radius must be positive, got {value}")
        self._radius = value

    @property
    def area(self):
        return math.pi * self._radius ** 2

    @property
    def diameter(self):
        return 2 * self._radius

c = Circle(5)
print(f"r={c.radius}, d={c.diameter}, area={c.area:.2f}")
c.radius = 10
print(f"r={c.radius}, d={c.diameter}, area={c.area:.2f}")
try:
    c.radius = -1
except ValueError as e:
    print(f"ValueError: {e}")

In [ ]:
# Exercise 3 Solution: Vector
import math

class Vector:
    def __init__(self, x, y):
        self._x = float(x)
        self._y = float(y)

    @property
    def x(self):
        return self._x

    @property
    def y(self):
        return self._y

    @property
    def magnitude(self):
        return math.hypot(self._x, self._y)

    @property
    def normalized(self):
        m = self.magnitude
        if m == 0:
            raise ValueError("Cannot normalize zero vector")
        return Vector(self._x / m, self._y / m)

    @classmethod
    def from_polar(cls, r, theta):
        """Create from polar coordinates (r, theta in radians)."""
        return cls(r * math.cos(theta), r * math.sin(theta))

    @staticmethod
    def dot(v1, v2):
        return v1.x * v2.x + v1.y * v2.y

    def __add__(self, other):
        return Vector(self._x + other.x, self._y + other.y)

    def __mul__(self, scalar):
        return Vector(self._x * scalar, self._y * scalar)

    def __repr__(self):
        return f"Vector({self._x:.3f}, {self._y:.3f})"

v1 = Vector(3, 4)
v2 = Vector.from_polar(5, math.pi / 4)
print(v1, "magnitude:", v1.magnitude)
print(v1.normalized)
print(v1 + v2)
print("dot product:", Vector.dot(v1, v2))

## 11. Mini-Project: Temperature Converter

A `Temperature` class where all three scales (Celsius, Fahrenheit, Kelvin) are always in sync via `@property`, with multiple `@classmethod` constructors and validation via `@staticmethod`.

In [ ]:
class Temperature:
    """
    Represents a temperature. Internally stored in Celsius.
    Access in any scale via properties; set via any scale too.
    """
    ABSOLUTE_ZERO_C = -273.15

    def __init__(self, celsius):
        self.celsius = celsius   # goes through setter for validation

    # --- Core property: celsius is the source of truth ---
    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if not Temperature.is_valid_celsius(value):
            raise ValueError(
                f"{value}°C is below absolute zero ({self.ABSOLUTE_ZERO_C}°C)"
            )
        self._celsius = float(value)

    # --- Fahrenheit: computed from celsius ---
    @property
    def fahrenheit(self):
        return self._celsius * 9/5 + 32

    @fahrenheit.setter
    def fahrenheit(self, value):
        self.celsius = (value - 32) * 5/9   # convert and store as celsius

    # --- Kelvin: computed from celsius ---
    @property
    def kelvin(self):
        return self._celsius - self.ABSOLUTE_ZERO_C

    @kelvin.setter
    def kelvin(self, value):
        self.celsius = value + self.ABSOLUTE_ZERO_C

    # --- Alternative constructors ---
    @classmethod
    def from_fahrenheit(cls, fahrenheit):
        """Create Temperature from Fahrenheit value."""
        return cls((fahrenheit - 32) * 5/9)

    @classmethod
    def from_kelvin(cls, kelvin):
        """Create Temperature from Kelvin value."""
        return cls(kelvin + Temperature.ABSOLUTE_ZERO_C)

    # --- Static validation utilities ---
    @staticmethod
    def is_valid_celsius(value):
        return value >= Temperature.ABSOLUTE_ZERO_C

    @staticmethod
    def is_valid_kelvin(value):
        return value >= 0

    @staticmethod
    def celsius_to_fahrenheit(c):
        return c * 9/5 + 32

    def __repr__(self):
        return (
            f"Temperature({self.celsius:.2f}°C / "
            f"{self.fahrenheit:.2f}°F / "
            f"{self.kelvin:.2f}K)"
        )

    def __eq__(self, other):
        return abs(self.celsius - other.celsius) < 1e-9

    def __lt__(self, other):
        return self.celsius < other.celsius


# --- Demo ---
print("=" * 50)
print("Temperature Converter Demo")
print("=" * 50)

boiling = Temperature(100)
freezing = Temperature(0)
body = Temperature.from_fahrenheit(98.6)
absolute_zero = Temperature.from_kelvin(0)

for t in [absolute_zero, freezing, body, boiling]:
    print(t)

print()
print("Modifying boiling point via Fahrenheit setter:")
boiling.fahrenheit = 50   # set in Fahrenheit, reads in all scales
print(boiling)

print()
try:
    invalid = Temperature(-300)
except ValueError as e:
    print(f"ValueError: {e}")

print()
print("Validation:")
print("Is 0K valid kelvin?", Temperature.is_valid_kelvin(0))
print("Is -1K valid kelvin?", Temperature.is_valid_kelvin(-1))